In [ ]:
from sentence_transformers import SentenceTransformer, util
import torch

model = SentenceTransformer('all-MiniLM-L6-v2')

model.to('cuda')

In [ ]:
import pandas as pd
df = pd.read_csv('django_data.csv')

embeddings = model.encode(df['Content'].tolist(), convert_to_tensor=True)

print(embeddings.shape)

In [3]:
def ask_question(question, certainty_threshold=0.35):

    question_embedding = model.encode(question, convert_to_tensor=True)

    cosine_scores = util.cos_sim(question_embedding, embeddings)[0]

    best_id = cosine_scores.argmax().item()
    certainity = cosine_scores[best_id].item()

    if certainity < certainty_threshold:
        return "Nie jestem pewien, czy to jest odpowiedź na Twoje pytanie. Spróbuj sformułować je inaczej."
    
    theme = df.iloc[best_id]['Topic']
    content = df.iloc[best_id]['Content']
    code = df.iloc[best_id]['Code']

    return f"Temat: {theme}\n\nTreść: {content}\n\nKod:\n{code}"
    

In [ ]:
print(ask_question("co to multiple inheritance?"))

In [ ]:
import ollama

response = ollama.chat(model='llama3.2:3b', messages=[
    {
        'role': 'user',
        'content': 'Czy znasz Django.'
    },
])
print(response['message']['content'])